In [45]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline, Trainer, TrainingArguments
from peft import get_peft_model, PeftModel, TaskType, PromptTuningConfig, PromptTuningInit

In [46]:
dataset=pd.read_csv("new_dataset_2.csv")
columns=dataset.columns
print(columns)
print("="*100)
for i in columns:
  print(dataset[i].unique().tolist())
print("="*100)

Index(['reviews', 'label'], dtype='object')
['Overall, I would describe this movie as average. The cinematography was pleasant and a few scenes were memorable, but the screenplay was fairly predictable and the ending was only satisfactory. The acting was mostly fine and the story was easy to follow, but there was nothing especially original or emotionally powerful. My final reaction is based on the complete viewing experience rather than only one scene or performance.', 'I was disappointed with this movie overall. Although there were a few decent moments, the weak writing, repetitive scenes, and lack of strong character development reduced the overall quality. The performances were inconsistent, the screenplay was predictable, and the ending felt rushed and unsatisfying. The final result matched the overall quality of the film.', 'This movie was neither very good nor very bad for me. I liked certain characters and a few scenes, while other parts felt repetitive, so my overall reaction 

In [47]:
dataset["label"]=dataset["label"].map({"neutral":0,"negative":2,"positive":1})
label2id={"neutral":0,"negative":2,"positive":1}
id2label={0:"neutral",2:"negative",1:"positive"}
print(dataset["label"].unique().tolist())

[0, 2, 1]


In [48]:
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")
model=AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=3,id2label=id2label)
promtp_tuning_config=PromptTuningConfig(task_type=TaskType.SEQ_CLS,prompt_tuning_init=PromptTuningInit.TEXT,prompt_tuning_init_text="classify the given sentences into positive, negative, neutral",num_virtual_tokens=10,tokenizer_name_or_path="bert-base-uncased")
peft_model=get_peft_model(model,promtp_tuning_config)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [49]:
data=tokenizer(dataset["reviews"].tolist(),padding="max_length",truncation=True,max_length=404)
input_ids=data["input_ids"]
attention_mask=data["attention_mask"]
token_type_ids=data["token_type_ids"]
dataset.drop(columns=["reviews"],inplace=True)
dataset["input_ids"]=input_ids
dataset["token_type_ids"]=token_type_ids
dataset["attention_mask"]=attention_mask
dataset=dataset[["input_ids","token_type_ids","attention_mask","label"]]
train_dataset=dataset.sample(frac=0.8).to_dict(orient="records")
test_dataset=dataset.sample(frac=0.2).to_dict(orient="records")

In [50]:
training_arguments=TrainingArguments(output_dir="./saved_model",num_train_epochs=3,learning_rate=1e-5)
trainer=Trainer(model=peft_model,args=training_arguments,train_dataset=train_dataset,eval_dataset=test_dataset)
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=90, training_loss=1.1259799533420138, metrics={'train_runtime': 2702.6088, 'train_samples_per_second': 0.266, 'train_steps_per_second': 0.033, 'total_flos': 149485336807680.0, 'train_loss': 1.1259799533420138, 'epoch': 3.0})

In [51]:
peft_model.save_pretrained("./prompt_tuning/model")
tokenizer.save_pretrained("./prompt_tuning/model")

('./prompt_tuning/model/tokenizer_config.json',
 './prompt_tuning/model/tokenizer.json')

In [59]:
from transformers import pipeline
peft_model=PeftModel(model,promtp_tuning_config).from_pretrained(model,"./prompt_tuning/model")
classifier=pipeline("sentiment-analysis",model=peft_model,tokenizer="./prompt_tuning/model")
review="Irumudi Movie Was Ok, Centimental"
classifier(review)

[{'label': 'negative', 'score': 0.4062076807022095}]